> **GitHub管理版（2026-11-05移管）**
> 既存の実験コードを移管したものです。セル出力は除去しており、今回の移管では実験を再実行していません。入力データ・保存済み結果は従来のDrive等を参照します。実行条件、認証、保存先を確認してから実行してください。元ファイルと用途は [ノートブック一覧](https://github.com/ryy1210/RMT_utils/blob/main/notebooks/README.md) を参照してください。

# 11 — モジュール全層のdecile削除 × DE
**本番未実行。** Llama・Gemmaそれぞれの7モジュール（UP, DOWN, GATE, Q, K, V, O）×10群×DE有無＝140評価＋baseline 1回。実行ボタンは利用者が押す。

Staats et al. (2025) §5の「各行列の特異値を10等分し、同じモジュールの全ブロックから同一順位帯を削除する」方式に対応する実験設計。モデルはLlama-3.2-3BとGemma-3-4B-PTであり、論文のモデル設定の完全再現ではない。

**decile 1＝最小側10%、decile 10＝最大側10%。** `np.array_split`で昇順の順位を10群に分ける。割り切れない場合は各群の個数差が最大1となり、DE有無で同じ個数。特異値が同率でも値でなく順位で割り当てる。

DEなし：元行列から該当成分を削除。DEあり：DE後の該当成分を元の座標へ逆変換して、元行列から差し引く。全成分を作り直す丸め誤差を避け、削除成分（約10%）のみを再構成する。DE前後で同じ順位でも特異方向は異なる。

各条件後に全対象行列を復元。他のモジュール・decileの削除を累積しない。MP閾値・bulk判定・LoRAは使わない。約90%のrankを残すため、圧縮効率ではなく削除感度を測る実験。

CPUの軽量テスト後、ColabのBF16対応GPU（L4で開始可能）で実行する。A100は必須ではない。140回の全量PPL評価は依然重い。キャッシュはランタイムのローカルディスクに保存し、セッションが消えると未完了モジュールではSVDを再計算する。Driveには逐次評価結果を保存。

In [ ]:
from pathlib import Path
import os, sys, subprocess, getpass
INSTALL_DEPENDENCIES = False
CLONE_IF_MISSING = True
REPO = Path('/content/RMT_utils') if Path('/content').exists() else Path.home() / 'GitHub/RMT_utils'
if not (REPO / 'funcs1.py').exists():
    if not CLONE_IF_MISSING:
        raise FileNotFoundError(f'RMT_utils not found: {REPO}')
    subprocess.run(['git', 'clone', 'https://github.com/ryy1210/RMT_utils', str(REPO)], check=True)
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO/'requirements-notebooks.txt')], check=True)
sys.path.insert(0, str(REPO))
# 必要なときだけ入力。キーをセル出力・設定ファイルに保存しない。
if not os.environ.get('HF_TOKEN'):
    token = getpass.getpass('HF_TOKEN (cached loginなら空欄): ')
    if token: os.environ['HF_TOKEN'] = token
print('Repository:', REPO)

In [ ]:
import json, hashlib, math, gc, time, re, platform, importlib.metadata, inspect, unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import funcs1
from funcs1 import _dyson_from_svd
MOUNT_DRIVE = True
if MOUNT_DRIVE and Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
if Path('/content').exists():
    DATA_ROOT=Path('/content/drive/MyDrive/TUS/hashiguchi/data')
else:
    base=Path.home()/'Library/CloudStorage/GoogleDrive-ryoya.zushi1210@gmail.com'
    candidates=[p/'TUS/hashiguchi/data' for p in base.iterdir() if unicodedata.normalize('NFC',p.name)=='マイドライブ']
    if len(candidates)!=1: raise FileNotFoundError('DATA_ROOTをDriveのdataフォルダに設定してください')
    DATA_ROOT=candidates[0]
LEGACY_OUTPUT=DATA_ROOT/'ibis2026_de_lra'/'ibis2026-de-controlled-v2-bulk-order'
OUTPUT=DATA_ROOT/'ibis2026_de_lra'/'decile-module-de-v1'
MODELS={'llama':'meta-llama/Llama-3.2-3B','gemma':'google/gemma-3-4b-pt'}
# 実行対象のみ変更可能。MODELSは固定し、両モデルの保存結果を同じ設定で保持する。
RUN_MODELS=['llama', 'gemma']  # Gemmaのみ: ['gemma']、両方: ['llama','gemma']
GROUPS={'UP':['up_proj'],'DOWN':['down_proj'],'GATE':['gate_proj'],
        'Q':['q_proj'],'K':['k_proj'],'V':['v_proj'],'O':['o_proj']}
DECILES=list(range(1,11))
CACHE_ROOT=Path('/content/rmt11_svd_cache') if Path('/content').exists() else Path('/tmp/rmt11_svd_cache')
DELETE_CACHE_AFTER_GROUP=True
RUN_EXPERIMENTS=True
ALL_REQUESTED_COMPLETED=False
SEQ_LEN=1024; BATCH_SIZE=1
SVD_DEVICE='cuda'
AUTO_DISCONNECT=False


## 固定評価データ・モデルrevision
08_2と同じ保存済みトークン列・revisionを使用。baselineはこの実験の環境で1回評価する。

In [ ]:
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def digest(x): return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def write_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False));tmp.replace(path)
def safe_exp(x): return math.exp(x) if x<709 else float('inf')
def atomic_csv(path,df):
    tmp=path.with_suffix('.tmp.csv');df.to_csv(tmp,index=False);tmp.replace(path)
manifest_path=LEGACY_OUTPUT/'manifest.json'
if not manifest_path.exists(): raise FileNotFoundError(f'08のmanifestがありません: {manifest_path}')
legacy=json.loads(manifest_path.read_text())['settings']
REVISIONS=legacy['revisions'];TOTAL=legacy['total']
assert all(re.fullmatch('[0-9a-f]{40}',REVISIONS[k]) for k in MODELS)
TOKENS={};TOKEN_HASH={}
for key in MODELS:
    assert legacy['models'][key]==MODELS[key]
    path=LEGACY_OUTPUT/f'{key}_tokens.npy'
    receipt=json.loads((LEGACY_OUTPUT/f'{key}_tokens.json').read_text())
    assert receipt['sha256']==sha(path), '08 token checksum mismatch'
    TOKENS[key]=np.load(path)
    assert TOKENS[key].ndim==2 and TOKENS[key].shape[1]==SEQ_LEN
    TOKEN_HASH[key]=sha(path)
OUTPUT.mkdir(parents=True,exist_ok=True)
print('固定token blocks:',{k:len(v) for k,v in TOKENS.items()})

In [ ]:
def decompose(weight):
    transposed=weight.shape[0]>weight.shape[1]
    w=(weight.T if transposed else weight).float().to(SVD_DEVICE)
    u,s,v=torch.linalg.svd(w,full_matrices=False)
    # 既存DEと同じ推定式。GPU thin SVD結果をfloat64で渡す。
    y,x,z=_dyson_from_svd(w.cpu().double().numpy(),u.cpu().double().numpy(),s.cpu().double().numpy(),v.cpu().double().numpy())
    ud,sd,vd=torch.linalg.svd(torch.as_tensor(y,device=SVD_DEVICE,dtype=torch.float32),full_matrices=False)
    return {False:(u,s,v,None,None,transposed),True:(ud,sd,vd,x,z,transposed)}
def reconstruct(parts,indices):
    u,s,v,x,z,t=parts
    idx=torch.as_tensor(indices,device=s.device,dtype=torch.long)
    result=(u[:,idx]*s[idx])@v[idx,:]
    if x is not None:
        result*=torch.as_tensor(np.sqrt(x),device=result.device,dtype=result.dtype)[:,None]
        result*=torch.as_tensor(np.sqrt(z),device=result.device,dtype=result.dtype)[None,:]
    if t: result=result.T
    if not torch.isfinite(result).all(): raise FloatingPointError('Nonfinite reconstruction')
    return result


In [ ]:
GEMMA_LOADER_VERSION='conditional_verified_v2'
def result_root(key): return OUTPUT/key
def check_loading_info(info):
    if not isinstance(info,dict): raise RuntimeError('Loading report missing')
    errors={k:info.get(k) for k in ['missing_keys','unexpected_keys','mismatched_keys','error_msgs'] if info.get(k)}
    if errors: raise RuntimeError(f'Pretrained checkpoint did not load completely: {errors}')

def extract_gemma_text(full):
    from transformers import Gemma3ForCausalLM
    # 正式クラスでロードした実Tensorを共有。meta上の空モデルへ全モジュールを置換する。
    decoder=full.model.language_model
    with torch.device('meta'):
        text=Gemma3ForCausalLM(full.config.text_config)
    text.model=decoder
    text.lm_head=full.lm_head
    text.generation_config=full.generation_config
    if any(p.is_meta for p in text.parameters()) or any(b.is_meta for b in text.buffers()):
        raise RuntimeError('Unloaded meta tensors remain')
    return text

def model_load(key):
    kwargs=dict(revision=REVISIONS[key],token=os.environ.get('HF_TOKEN'),torch_dtype=torch.bfloat16,
                low_cpu_mem_usage=True,output_loading_info=True)
    if key=='gemma':
        from transformers import Gemma3ForConditionalGeneration
        # 修正: checkpointのarchitectureと一致するクラスを使う。text_configだけを渡さない。
        full,info=Gemma3ForConditionalGeneration.from_pretrained(MODELS[key],**kwargs)
        check_loading_info(info)
        model=extract_gemma_text(full)
        del full; gc.collect()
    else:
        model,info=AutoModelForCausalLM.from_pretrained(MODELS[key],**kwargs)
        check_loading_info(info)
    actual=sum(p.numel() for p in model.parameters())
    if actual!=TOTAL[key]: raise ValueError(f'Parameter count mismatch {actual} != {TOTAL[key]}')
    if any(p.is_meta for p in model.parameters()): raise RuntimeError('Unloaded model parameters')
    root=result_root(key); root.mkdir(parents=True,exist_ok=True)
    write_json(root/'loading_report.json',dict(model=MODELS[key],revision=REVISIONS[key],
        loader_version=GEMMA_LOADER_VERSION if key=='gemma' else 'auto_verified_v2',
        transformers=importlib.metadata.version('transformers'),parameter_count=actual,
        missing_keys=[],unexpected_keys=[],mismatched_keys=[],config_hash=CONFIG_HASH))
    model.eval(); model.config.use_cache=False
    return model.to('cuda')

def evaluate(model,tokens):
    losses=[]; counts=[]
    for start in range(0,len(tokens),BATCH_SIZE):
        ids=torch.as_tensor(tokens[start:start+BATCH_SIZE],device=next(model.parameters()).device)
        logits=model(input_ids=ids,use_cache=False).logits
        sums=torch.zeros(ids.shape[0],device=ids.device,dtype=torch.float64)
        for pos in range(0,ids.shape[1]-1,64):
            end=min(pos+64,ids.shape[1]-1)
            loss=F.cross_entropy(logits[:,pos:end].float().transpose(1,2),ids[:,pos+1:end+1],reduction='none')
            if not torch.isfinite(loss).all(): raise FloatingPointError('Nonfinite NLL')
            sums+=loss.double().sum(1)
        losses.extend(sums.cpu().tolist()); counts.extend([ids.shape[1]-1]*ids.shape[0]); del logits
    return np.asarray(losses),np.asarray(counts)

def resolve(model,name):
    found=[m for k,m in model.named_modules() if isinstance(m,torch.nn.Linear) and (k==name or k.endswith('.'+name))]
    if len(found)!=1: raise ValueError(f'Ambiguous module {name}')
    return found[0]

In [ ]:
evaluate=torch.inference_mode()(evaluate)
def function_fingerprint(f):
    # ipykernelの一時filename・セル番号は再起動で変わるのでhashに含めない。
    import types
    def normalize(code):
        return [code.co_code.hex(),code.co_names,code.co_varnames,
                [normalize(x) if isinstance(x,types.CodeType) else repr(x) for x in code.co_consts]]
    return digest(normalize(f.__code__))

def read_result(path):
    if not (path/'summary.json').exists(): return None
    obj=json.loads((path/'summary.json').read_text())
    assert obj['config_hash']==CONFIG_HASH
    f=path/'losses.npz'
    assert sha(f)==obj['loss_hash'], 'Result checksum mismatch'
    with np.load(f) as z: loss=z['loss'];count=z['count']
    assert len(loss)==len(TOKENS[obj['model']]) and np.all(count==SEQ_LEN-1) and np.isfinite(loss).all()
    return obj

def save_result(path,key,loss,count,extra):
    path.mkdir(parents=True,exist_ok=True)
    with open(path/'losses.tmp','wb') as f: np.savez_compressed(f,loss=loss,count=count)
    (path/'losses.tmp').replace(path/'losses.npz')
    value=safe_exp(float(loss.sum()/count.sum()))
    obj=dict(model=key,config_hash=CONFIG_HASH,ppl=value if np.isfinite(value) else None,
             ppl_overflow=not np.isfinite(value),loss_hash=sha(path/'losses.npz'),**extra)
    write_json(path/'summary.json',obj)
    return obj

## 順位分割・再開・全層同時削除
設定・関数の指紋を保存する。設定が異なれば同じ出力先への混在を拒否する。完了済み評価は保存損失のチェックサムを確認して再利用。全140条件が完了すればモデルも読み込まない。

SVDキャッシュの容量を事前検査。重みは評価時にはdenseのままで、約90%rankの因子化によるパラメータ削減・高速化は主張しない。

In [ ]:
def decile_indices(p,decile):
    if p<10 or decile not in DECILES:raise ValueError('Need p>=10 and decile 1..10')
    # torch.linalg.svdは降順。最小側から順に群を割り当てる。
    return np.array_split(np.arange(p-1,-1,-1),10)[decile-1]

def remove_decile(original,parts,decile):
    idx=decile_indices(len(parts[1]),decile)
    removed=reconstruct(parts,idx)
    value=original.to(device=removed.device,dtype=torch.float32)-removed
    if not torch.isfinite(value).all():raise FloatingPointError('Nonfinite pruned weight')
    return value,idx

def cache_path(key,group,name):
    return CACHE_ROOT/CONFIG_HASH/key/group/(digest(name)[:24]+'.pt')

def prepare_svd_cache(key,group,matrices):
    import shutil
    root=CACHE_ROOT/CONFIG_HASH/key/group;root.mkdir(parents=True,exist_ok=True)
    required=sum(2*4*(min(m.weight.shape)*(sum(m.weight.shape)+1)+sum(m.weight.shape)) for _,m in matrices)
    existing=sum(p.stat().st_size for p in root.glob('*.pt'))
    if shutil.disk_usage(root).free < max(0,required-existing)+1024**3:
        raise RuntimeError(f'SVD cache requires about {required/1e9:.1f} GB; insufficient free disk')
    for i,(name,module) in enumerate(matrices):
        path=cache_path(key,group,name);receipt=path.with_suffix('.json')
        if path.exists() and receipt.exists():
            obj=json.loads(receipt.read_text())
            assert obj['config_hash']==CONFIG_HASH and obj['name']==name and obj['sha256']==sha(path)
            assert obj['shape']==list(module.weight.shape)
            continue
        print(f'{key} {group} SVD {i+1}/{len(matrices)}: {name}',flush=True)
        original=module.weight.detach().cpu().clone();parts=decompose(original)
        stored={}
        for de,(u,s,v,x,z,t) in parts.items():
            stored[de]=(u.cpu(),s.cpu(),v.cpu(),None if x is None else torch.as_tensor(np.asarray(x).copy()),
                        None if z is None else torch.as_tensor(np.asarray(z).copy()),t)
        tmp=path.with_suffix('.tmp');torch.save(stored,tmp);tmp.replace(path)
        write_json(receipt,dict(config_hash=CONFIG_HASH,name=name,shape=list(original.shape),sha256=sha(path)))
        del original,parts,stored;gc.collect();torch.cuda.empty_cache()
    return root

def load_parts(path,de,device):
    obj=torch.load(path,map_location='cpu',weights_only=True)
    u,s,v,x,z,t=obj[de]
    return u.to(device),s.to(device),v.to(device),None if x is None else x.numpy(),None if z is None else z.numpy(),t

def validate_group(model,targets):
    matrices=[(n,m) for n,m in model.named_modules() if isinstance(m,torch.nn.Linear) and n.split('.')[-1] in targets]
    for target in targets:
        names=[n for n,m in matrices if n.split('.')[-1]==target]
        blocks=[int(re.search(r'layers\.(\d+)\.',n).group(1)) for n in names]
        assert sorted(blocks)==list(range(model.config.num_hidden_layers)), f'Incomplete module: {target}'
    return matrices

def run_group(model,key,group,baseline,todo):
    import shutil
    matrices=validate_group(model,GROUPS[group])
    root=prepare_svd_cache(key,group,matrices)
    originals={name:module.weight.detach().cpu().clone() for name,module in matrices}
    try:
        for de,decile in todo:
            dest=OUTPUT/key/'groups'/group/f'{int(de)}_decile{decile:02d}'
            records=[];start=time.time()
            try:
                for name,module in matrices:
                    parts=load_parts(cache_path(key,group,name),de,module.weight.device)
                    approx,idx=remove_decile(originals[name],parts,decile)
                    p=len(parts[1]);r=p-len(idx)
                    cast=approx.to(module.weight.dtype)
                    assert torch.isfinite(cast).all()
                    with torch.no_grad():module.weight.copy_(cast)
                    records.append(dict(name=name,de=de,decile=decile,p=p,removed=len(idx),retained_rank=r,
                        removed_fraction=len(idx)/p,descending_indices=idx.tolist(),
                        relative_frobenius_error=float(torch.linalg.vector_norm(approx.cpu()-originals[name].float())/
                                                       torch.linalg.vector_norm(originals[name].float()))))
                    del parts,approx,cast
                loss,count=evaluate(model,TOKENS[key])
                dest.mkdir(parents=True,exist_ok=True)
                write_json(dest/'matrices.json',dict(config_hash=CONFIG_HASH,records=records))
                obj=save_result(dest,key,loss,count,dict(group=group,de=de,decile=decile,matrix_count=len(matrices),
                    baseline_ppl=baseline['ppl'],sum_removed=sum(r['removed'] for r in records),
                    sum_rank=sum(r['retained_rank'] for r in records),matrix_plan_hash=sha(dest/'matrices.json'),seconds=time.time()-start))
                print(f'SAVED {key} {group} decile={decile} DE={de} PPL={obj["ppl"]}',flush=True)
            finally:
                # 全条件・例外・中断で元の全対象重みに戻す。
                with torch.no_grad():
                    for name,module in matrices:module.weight.copy_(originals[name].to(module.weight.device))
        if DELETE_CACHE_AFTER_GROUP:shutil.rmtree(root)
    finally:
        del originals;gc.collect();torch.cuda.empty_cache()

def initialize_manifest():
    global CONFIG_HASH
    settings=dict(version='11-decile-v1',models=MODELS,revisions=REVISIONS,total=TOTAL,token_hash=TOKEN_HASH,
        groups=GROUPS,deciles=DECILES,partition='ascending array_split, small first',seq_len=SEQ_LEN,
        batch=BATCH_SIZE,dtype='bfloat16',svd_device=SVD_DEVICE,funcs1_hash=sha(Path(funcs1.__file__)),
        implementation={f.__name__:function_fingerprint(f) for f in [decompose,reconstruct,decile_indices,remove_decile,
            model_load,extract_gemma_text,check_loading_info,evaluate.__wrapped__,read_result,save_result,
            prepare_svd_cache,load_parts,validate_group,run_group,verified_job,run]})
    CONFIG_HASH=digest(settings);path=OUTPUT/'manifest.json'
    if path.exists() and json.loads(path.read_text())['config_hash']!=CONFIG_HASH:
        raise ValueError('設定/実装が変更されています。以前の設定に戻すか新しいOUTPUTを使用してください')
    write_json(path,dict(config_hash=CONFIG_HASH,settings=settings))

def verified_job(key,group,de,decile):
    path=OUTPUT/key/'groups'/group/f'{int(de)}_decile{decile:02d}'
    obj=read_result(path)
    if obj:
        assert obj['model']==key and obj['group']==group and obj['de']==de and obj['decile']==decile
        assert sha(path/'matrices.json')==obj['matrix_plan_hash']
    return obj

def run():
    global ALL_REQUESTED_COMPLETED
    ALL_REQUESTED_COMPLETED=False
    if not RUN_MODELS or len(RUN_MODELS)!=len(set(RUN_MODELS)) or not set(RUN_MODELS)<=set(MODELS):
        raise ValueError('RUN_MODELS must contain llama and/or gemma without duplicates')
    initialize_manifest()
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():raise RuntimeError('BF16対応CUDA GPUが必要です')
    write_json(OUTPUT/'environment.json',dict(gpu=torch.cuda.get_device_name(),python=platform.python_version(),
        versions={p:importlib.metadata.version(p) for p in ['torch','transformers','numpy','datasets']}))
    for key in RUN_MODELS:
        todo={group:[(de,d) for de in [False,True] for d in DECILES if verified_job(key,group,de,d) is None] for group in GROUPS}
        baseline=read_result(OUTPUT/key/'baseline')
        if baseline and not any(todo.values()):print(key,'全条件完了：モデル・SVD・PPLをスキップ');continue
        model=model_load(key)
        try:
            if baseline is None:
                loss,count=evaluate(model,TOKENS[key]);baseline=save_result(OUTPUT/key/'baseline',key,loss,count,dict(kind='baseline'))
            if baseline['ppl'] is None or baseline['ppl']>1000:raise RuntimeError('Baseline異常')
            for group,remaining in todo.items():
                if not remaining:print(group,'完了済み：SVD/PPLスキップ');continue
                print(key,group,'残り',len(remaining),'条件',flush=True)
                run_group(model,key,group,baseline,remaining)
        finally:del model;gc.collect();torch.cuda.empty_cache()
    ALL_REQUESTED_COMPLETED=True


## 軽量チェックと本番
初回はRUN_EXPERIMENTS=Trueで軽量チェックのみ。GPUと入力を確認後Trueにして「すべて実行」。再開時も同じ設定・保存先で実行する。

`RUN_MODELS=["llama"]`が既定。Gemmaは`["gemma"]`、両モデル連続は`["llama","gemma"]`に変更。1モデルにつき140条件＋baseline、両モデルで280条件＋baseline 2回。実行対象の切替だけでは設定ハッシュを変えないため、同じOUTPUTで再開可能。Gemmaは画像部分を使用せず、09_2と同じ検証済みtextモデル抽出方法を使う。両モデルの保存済み評価トークン・revisionを読み込むため、08_2のデータが必要。

In [ ]:
def smoke_test():
    global SVD_DEVICE
    old=SVD_DEVICE;SVD_DEVICE='cpu'
    try:
        torch.manual_seed(17)
        for shape in [(24,80),(80,24),(24,24)]:
            w=torch.randn(*shape);parts=decompose(w);p=min(shape)
            bins=[decile_indices(p,d) for d in DECILES]
            assert np.array_equal(np.sort(np.concatenate(bins)),np.arange(p))
            assert max(map(len,bins))-min(map(len,bins))<=1
            assert bins[0][0]==p-1 and bins[-1][-1]==0
            for de in [False,True]:
                assert torch.allclose(reconstruct(parts[de],np.arange(p)),w,atol=4e-5,rtol=4e-5)
                for d,idx in zip(DECILES,bins):
                    got,removed=remove_decile(w,parts[de],d)
                    expected=reconstruct(parts[de],np.setdiff1d(np.arange(p),idx))
                    assert torch.allclose(got,expected,atol=4e-5,rtol=4e-5)
        print('PASS: 10群分割・逆変換・転置・全20条件の削除式')
    finally:SVD_DEVICE=old
smoke_test()
if RUN_EXPERIMENTS:run()
else:print('本番未実行。RUN_MODELS=',RUN_MODELS,'; RUN_EXPERIMENTS=Trueで各モデル140条件＋baselineを実行。')


## 集計と可視化
PPL自体とbaselineからのPPL増分を表示する。decile 1の削除による悪化がDE後に小さくなるか、同じモジュールの他の群との順位がどう変わるかを確認する。DEによるPPL低下率が正ならDE有利。これは個々の特異方向の情報移動を直接示すものではない。

欠損は未完了として表示する。全10群が揃った場合だけ削除感度の順位を計算する。PPLが数値範囲を超えた場合はoverflowとして区別する。

In [ ]:
initialize_manifest()
rows=[]
for key in MODELS:
    for group in GROUPS:
        for de in [False,True]:
            for d in DECILES:
                obj=verified_job(key,group,de,d)
                rows.append(dict(**obj,status='complete') if obj else dict(model=key,group=group,de=de,decile=d,ppl=None,status='pending'))
RESULTS=pd.DataFrame(rows);atomic_csv(OUTPUT/'results.csv',RESULTS)
display(RESULTS.groupby(['model','status']).size().rename('conditions').reset_index())
done=RESULTS[RESULTS.status=='complete'].copy()
if len(done):
    done['ppl']=pd.to_numeric(done.ppl,errors='coerce')
    done['ppl_increase']=done.ppl-done.baseline_ppl
    if done.ppl.isna().any():print('注意：PPL overflowあり。欠損を性能良好と解釈しないでください。')
    pairs=done.pivot(index=['model','group','decile'],columns='de',values='ppl').reindex(columns=[False,True])
    pairs.columns=['ppl_raw','ppl_de'];pairs=pairs.reset_index()
    pairs['DE_gain_percent']=100*(1-pairs.ppl_de/pairs.ppl_raw)
    atomic_csv(OUTPUT/'paired.csv',pairs);display(pairs)
    ranks=[]
    for (key,group,de),g in done.groupby(['model','group','de']):
        if len(g)==10 and g.ppl.notna().all():
            g=g.copy();g['importance_rank']=g.ppl.rank(ascending=False,method='min');ranks.append(g)
    if ranks:atomic_csv(OUTPUT/'importance_ranks.csv',pd.concat(ranks,ignore_index=True))
    for key in MODELS:
        fig,axes=plt.subplots(2,4,figsize=(16,8))
        for ax,group in zip(axes.flat,GROUPS):
            for de in [False,True]:
                g=done[(done.model==key)&(done.group==group)&(done.de==de)].set_index('decile').reindex(DECILES)
                ax.plot(DECILES,g.ppl,marker='s' if de else 'o',label=f'DE={de}')
            baseline=read_result(OUTPUT/key/'baseline')
            if baseline:ax.axhline(baseline['ppl'],c='gray',ls='--',label='baseline')
            ax.set(title=group,xlabel='Decile (1=smallest, 10=largest)',ylabel='PPL',yscale='log',xticks=DECILES);ax.legend(fontsize=7)
        axes.flat[-1].axis('off');fig.tight_layout()
        fig.savefig(OUTPUT/f'{key}_decile_ppl.pdf');fig.savefig(OUTPUT/f'{key}_decile_ppl.png',dpi=160);plt.show()
    atomic_csv(OUTPUT/'results_with_baseline_delta.csv',done)
else:print('保存結果はまだありません。')


## 検証範囲と解釈
CPUの小行列で10群の重複・欠落なし、DE逆変換、縦長・横長・正方行列、同rank削除との一致を確認。2ブロックの模擬モデルで全20条件の保存・読戻し、DE前後の同数削除、キャッシュ再利用、正常時・例外時の重み復元を確認。本番のLlama/Gemmaを用いたGPU実行は未検証。

同一モジュールの全ブロックを同時に削除するため、層間の相互作用を含む感度を測る。DE前後の同じ順位帯は同じ特異方向ではない。下側の削除感度低下が確認されても、情報が上側に移動したことの直接証明ではない。各行列の10%は絶対個数・エネルギー量・モジュール総パラメータ数を揃えるものではない。

In [ ]:
if AUTO_DISCONNECT and RUN_EXPERIMENTS and ALL_REQUESTED_COMPLETED:
    from google.colab import runtime
    runtime.unassign()
